# Unit 15 — Adapt and evaluate a pretrained Diffusion Policy

This unit adapts a prepared policy on the frozen LIBERO training split. It saves the policy with its processors, reloads the saved bundle, and evaluates the base and adapted policies on the same development states.

The full run targets one GPU with up to 48 GB memory and a two-hour time limit. The configured update count is recorded. A two-step check is not training evidence.

In [ ]:
import copy
import json
import random
import signal
import sys
import time
from collections.abc import Callable, Mapping
from pathlib import Path
from statistics import fmean
from types import FrameType

import numpy as np
import torch
from torch import Tensor, nn
from torch.utils.data import default_collate

WORKSPACE = next(
    p
    for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (p / "manifests/act_references.yaml").is_file()
)
sys.path.insert(0, str(WORKSPACE / "src"))
from lerobot.configs import FeatureType, PolicyFeature
from lerobot.policies.diffusion import DiffusionConfig, DiffusionPolicy
from lerobot.policies.diffusion.modeling_diffusion import DiffusionConditionalUnet1d

from vla_lab.contracts import EpisodeOutcome
from vla_lab.evaluation.states import evaluate_states
from vla_lab.sim.episode import open_libero_task, run_policy_episode
from vla_lab.teaching.act_model import open_libero_dataset
from vla_lab.teaching.course_diffusion import learner_unet_state_for_lerobot
from vla_lab.teaching.diffusion import CallbackDiffusionPolicy, sample_actions
from vla_lab.training.diffusion import (
    diffusion_delta_timestamps,
    load_diffusion_bundle,
    save_diffusion_bundle,
    train_diffusion_reference,
)
from vla_lab.types import JSONValue

## Preserve the evaluation boundary

A prepared checkpoint contains model weights, action and state processors, training statistics, and a manifest for the frozen data split. Start from this exact bundle. Use training episodes for gradient updates. Keep validation and development episodes outside the optimizer.

Record baseline evaluation before updates. Fine-tune the U-Net, save periodic complete bundles, then reload the final bundle from disk. Evaluate the reloaded policy on the same fixed development states and seed. Compare task success and sampler time. A lower training loss alone does not show that the robot task improved.


## Exercise 15.1 — Build the learner U-Net

**Purpose.** Place the Unit 13 learner network in the prepared policy while preserving its trained weights.

**Contract.** `make_finetune_unet(prepared_unet)` takes the prepared policy's `DiffusionConditionalUnet1d` and returns the learner U-Net (`nn.Module`) with the same architecture and a copy of every prepared weight.

**Steps.** Read the prepared U-Net configuration. Build the same action width, conditioning width, stage widths, and block settings. Copy weights with strict key and shape checks. Before training, compare outputs on fixed inputs and require the largest absolute difference to be below `1e-4`. Keep the image encoder and processors in the policy.

In [ ]:
def make_finetune_unet(prepared_unet: DiffusionConditionalUnet1d) -> nn.Module:
    """Create the same-size learner U-Net and copy all prepared weights strictly."""
    raise NotImplementedError("Complete Exercise 15.1: make_finetune_unet")

In [ ]:
small_config = DiffusionConfig(
    input_features={"observation.state": PolicyFeature(FeatureType.STATE, (8,))},
    output_features={"action": PolicyFeature(FeatureType.ACTION, (7,))},
    n_obs_steps=2,
    horizon=16,
    n_action_steps=8,
    down_dims=(16, 32),
    diffusion_step_embed_dim=32,
    pretrained_backbone_weights=None,
    push_to_hub=False,
    device="cpu",
)
prepared = DiffusionConditionalUnet1d(small_config, global_cond_dim=6).eval()
learner = make_finetune_unet(prepared).eval()
x = torch.randn(2, 16, 7)
t = torch.tensor([5, 37])
cond = torch.randn(2, 6)
with torch.no_grad():
    gap = (prepared(x, t, global_cond=cond) - learner(x, t, global_cond=cond)).abs().max().item()
assert gap < 1e-4, f"Strict weight-copy parity failed: {gap}"
print(f"strict U-Net parity: max error {gap:.2e}")

## Full experiment — Fine-tune and evaluate

The selected full group needs a local prepared checkpoint, frozen LIBERO data, a simulator, and a CUDA GPU. The recipe uses 2,000 updates and batch size 8 as a starting point. Keep the run within 48 GB GPU memory and two hours. If memory does not fit, lower batch size and record the change. Do not report a short smoke test as adaptation training.

The experiment evaluates the base checkpoint first, trains the policy's learner U-Net on training episodes, saves the policy and processors, reloads them, and evaluates the same development states before and after with the Unit 14 sampler passed to CallbackDiffusionPolicy. Both runs use the same 20-step schedule and reset seeds. An interrupted or failed run does not count as complete.

In [ ]:
BASE_CHECKPOINT = (
    WORKSPACE / ".local-data/runs/diffusion-bowl-plate-reference-2000/checkpoint-002000"
)
CONFIGS = WORKSPACE / "configs"
DATA = WORKSPACE / ".local-data"
OUTPUT = DATA / "runs" / f"course-diffusion-adaptation-{int(time.time())}"
DEVICE = "cuda"
UPDATES = 2000
BATCH_SIZE = 8
FIXED_STATES = [10, 11, 12, 13, 14]

In [ ]:
base_policy, base_pre, base_post, base_manifest = load_diffusion_bundle(BASE_CHECKPOINT)
base_policy = base_policy.to(DEVICE)
assert base_manifest["task_alias"] == "bowl_plate"
VALIDATION = open_libero_dataset(
    CONFIGS, DATA, "bowl_plate", "validation", diffusion_delta_timestamps(base_policy.config, 10)
)
VALIDATION_ROWS = list(range(16))


def validation_epsilon(
    policy: DiffusionPolicy, processor: Callable[[Mapping[str, Tensor]], Mapping[str, Tensor]]
) -> float:
    raw = default_collate([VALIDATION[i] for i in VALIDATION_ROWS])
    batch = processor(raw)
    device = next(policy.parameters()).device
    batch = {
        key: value.to(device) if isinstance(value, torch.Tensor) else value
        for key, value in batch.items()
    }
    policy.eval()
    torch.manual_seed(20261006)
    with torch.no_grad():
        loss, _metrics = policy(batch)
    return float(loss)


validation_loss_before = validation_epsilon(base_policy, base_pre)

In [ ]:
def evaluate_with_learner_sampler(
    checkpoint: Path,
    policy: DiffusionPolicy,
    preprocessor: Callable[[Mapping[str, Tensor]], Mapping[str, Tensor]],
    postprocessor: Callable[[Mapping[str, Tensor]], Mapping[str, Tensor]],
    state_ids: list[int],
    output_dir: Path,
) -> tuple[dict[str, JSONValue], dict[int, float]]:
    """Evaluate one checkpoint with Unit 14 sampling inside the real policy callback."""
    manifest = json.loads((checkpoint / "manifest.json").read_text())
    sampler_times: dict[int, float] = {}

    def run_one(state_id: int) -> EpisodeOutcome:
        seed = 20260930 + state_id
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        runner = CallbackDiffusionPolicy(
            policy,
            sampler=sample_actions,
            num_inference_steps=policy.config.num_inference_steps,
            seed=seed,
        )
        environment = open_libero_task("bowl_plate", state_id, CONFIGS, DATA)
        try:
            result = run_policy_episode(
                environment,
                runner,
                preprocessor,
                postprocessor,
                "bowl_plate",
                state_id,
                CONFIGS,
                reset_seed=seed,
            )
        finally:
            environment.close()
        sampler_times[state_id] = fmean(runner.sample_seconds) * 1000
        return EpisodeOutcome(
            task_id="bowl_plate",
            initial_state_id=state_id,
            partition="development",
            policy_revision=str(manifest["model_sha256"]),
            success_predicate_version="libero-goal-v1",
            success=result.success,
            termination_reason=result.termination_reason,
            metrics={
                "steps": result.steps,
                "control_hz": result.control_hz,
                "simulated_duration_s": result.simulated_duration_s,
                "inference_wall_s": result.inference_wall_s,
                "clipped_values": result.clipped_values,
                "reset_seed": seed,
                "sampler_calls": runner.chunks_made,
                "sampler_mean_ms": sampler_times[state_id],
            },
            artifact_paths={},
        )

    summary = evaluate_states(
        configs_root=CONFIGS,
        task_id="bowl_plate",
        partition="development",
        state_ids=state_ids,
        policy_revision=str(manifest["model_sha256"]),
        output_dir=output_dir,
        run_one=run_one,
    )
    return summary, sampler_times

In [ ]:
before, before_sampler_ms = evaluate_with_learner_sampler(
    BASE_CHECKPOINT,
    base_policy,
    base_pre,
    base_post,
    FIXED_STATES,
    DATA / "evaluations" / f"course-diffusion-before-{int(time.time())}",
)

In [ ]:
PREPARED_UNET = base_policy.diffusion.unet
NATIVE_UNET = copy.deepcopy(PREPARED_UNET)
learner_unet = make_finetune_unet(PREPARED_UNET).to(DEVICE)
action_dim = base_policy.config.action_feature.shape[0]
first = PREPARED_UNET.down_modules[0][0]
global_dim = first.cond_encoder[1].in_features - base_policy.config.diffusion_step_embed_dim
probe = torch.randn(
    2, base_policy.config.horizon, action_dim, device=next(PREPARED_UNET.parameters()).device
)
timestep = torch.tensor([5, 37], device=probe.device)
condition = torch.zeros(2, global_dim, device=probe.device)
with torch.no_grad():
    difference = (
        (learner_unet(probe, timestep, condition) - PREPARED_UNET(probe, timestep, condition))
        .abs()
        .max()
        .item()
    )
assert difference < 1e-4, f"Strict checkpoint parity failed: {difference}"
base_policy.diffusion.unet = learner_unet

In [ ]:
assert torch.cuda.is_available(), "Adaptation requires CUDA."
assert torch.cuda.get_device_properties(0).total_memory <= 48 * 1024**3, (
    "The memory target is at most 48 GB."
)


def stop_adaptation(_signal: signal.Signals, _frame: FrameType | None) -> None:
    raise TimeoutError("Adaptation reached its two-hour limit; keep saved checkpoints.")


signal.signal(signal.SIGALRM, stop_adaptation)
signal.alarm(100 * 60)
try:
    result = train_diffusion_reference(
        task_alias="bowl_plate",
        configs_root=CONFIGS,
        data_root=DATA,
        output_dir=OUTPUT,
        steps=UPDATES,
        batch_size=BATCH_SIZE,
        checkpoint_every=500,
        device=DEVICE,
        policy=base_policy,
    )
finally:
    signal.alarm(0)
assert result.steps == UPDATES

In [ ]:
native_state = learner_unet_state_for_lerobot(learner_unet.state_dict())
NATIVE_UNET.load_state_dict(native_state, strict=True)
base_policy.diffusion.unet = NATIVE_UNET
FINAL_BUNDLE = OUTPUT / "reloadable-policy"
save_diffusion_bundle(base_policy, base_pre, base_post, FINAL_BUNDLE, "bowl_plate", CONFIGS)
reloaded, loaded_pre, loaded_post, final_manifest = load_diffusion_bundle(FINAL_BUNDLE)
assert loaded_pre is not None and loaded_post is not None
assert final_manifest["task_alias"] == base_manifest["task_alias"]
validation_loss_after = validation_epsilon(reloaded, loaded_pre)
after, after_sampler_ms = evaluate_with_learner_sampler(
    FINAL_BUNDLE,
    reloaded,
    loaded_pre,
    loaded_post,
    FIXED_STATES,
    DATA / "evaluations" / f"course-diffusion-after-{int(time.time())}",
)
print(
    {
        "updates": result.steps,
        "loss": result.last_loss,
        "validation_before": validation_loss_before,
        "validation_after": validation_loss_after,
        "success_before": before["success_rate"],
        "success_after": after["success_rate"],
        "mean_sampler_ms_before": fmean(before_sampler_ms.values()),
        "mean_sampler_ms_after": fmean(after_sampler_ms.values()),
    }
)

## Compare before and after

Record the checkpoint IDs, update count, seed, state IDs, success results, mean sampler time from the Unit 14 sampler, and peak GPU memory. A successful reload shows that the saved bundle is usable. Results on five development states describe those states only.

## What this unit established

Adaptation changes the U-Net on the training split, saves model weights and processors, reloads them, and evaluates before and after with a fixed protocol. State the exact hardware and completed update count.